<a href="https://colab.research.google.com/github/marry12256/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26_week3_maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
csv_path = "/content/drive/MyDrive/labels (2).csv"

In [18]:
!pip install transformers torch pillow pandas

In [19]:

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row["image"]).convert("RGB")

        # Process image for the model
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [20]:
import pandas as pd

df = pd.read_csv(csv_path)

# Update image paths
df["image"] = df["image"].str.replace(
    "data/raw/newspaper/",
    "/content/drive/MyDrive/processed_images/",
    regex=False
)

# Save updated CSV
df.to_csv(csv_path, index=False)

print("Image paths updated successfully!")
print(df.head())

Image paths updated successfully!
                                               image  \
0  /content/drive/MyDrive/processed_images/textbo...   
1  /content/drive/MyDrive/processed_images/textbo...   
2  /content/drive/MyDrive/processed_images/textbo...   
3  /content/drive/MyDrive/processed_images/textbo...   
4  /content/drive/MyDrive/processed_images/textbo...   

                                             text  
0       کمرے کل بنے گا، انہیں مفت میں جانے دی گئی  
1  ہوٹل میں کھانا کھلا ہے۔ کھانا اچھا ہے۔ سٹاف کا  
2        لوگ دو ستارے ہیں۔ کمرے صاف اور روشن ہیں۔  
3       ہوٹل کا سٹاف اچھا میں ہے۔ ہمیں بہت انتظار  
4     کروایا۔ میں دوبارہ اس ہوٹل میں نہیں آؤں گا۔  


In [21]:
# Create dataset
dataset = UrduOCRDataset(csv_path, processor)

# Test it loads correctly
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Create train / test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40
